# Phase 4: Final System Evaluation
**Objective:** Compare the integrated AI system (DQN Dispatch + MARL Traffic) against the Baseline system (Nearest-Idle + Standard Traffic).

We run a full day of simulated emergencies in Kigali. We query SUMO's live routing engine to determine the actual drive times through peak-hour traffic, combined with our dynamic hospital queuing model, to calculate the definitive `Total Time-to-Care` metric.

In [1]:
import sys
import json
import logging
import math
import traci
import sumolib
import numpy as np
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.environment.manager import SimulationManager
from src.environment.hospital import Hospital
from src.agents.dispatch_dqn import DispatchAgent
from src.baselines.dispatch_heuristics import BaselineDispatchers

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

net_path = Path("../data/processed/kigali_connected.net.xml")
route_path = Path("../data/processed/kigali_connected_traffic.rou.xml")
incidents_path = Path("../data/processed/incidents_seed42.json")
dqn_model_path = Path("../models/dqn_dispatch_v1.pt")

net = sumolib.net.readNet(str(net_path))

def get_nearest_edge(x, y):
    """Finds the closest drivable road with a massive 2km net and a crash-proof fallback."""
    # MASSIVE 2000m radius to account for the deleted side streets
    edges = net.getNeighboringEdges(x, y, 2000) 
    
    valid_edges = []
    for e in edges:
        edge_obj = e[0]
        # Skip the known broken one-way dead end
        if edge_obj.getID() == "1188083591":
            continue
        if edge_obj.allows("passenger"):
            valid_edges.append(edge_obj)
    
    if valid_edges:
        # Sort by length to snap to the main artery
        valid_edges.sort(key=lambda e: e.getLength(), reverse=True)
        return valid_edges[0].getID()
        
    # THE ULTIMATE FAILSAFE: If it STILL finds nothing, grab the first legal road in the city.
    # This guarantees 'h.edge_id' will never be None again.
    fallback_edges = [e for e in net.getEdges() if e.allows("passenger")]
    return fallback_edges[0].getID()

In [2]:
from enum import Enum
import pandas as pd

# --- STATE MACHINES ---
class IncidentState(Enum):
    PENDING = "PENDING"           
    CLAIMED = "CLAIMED"           
    REACHED = "REACHED"           
    RESOLVED = "RESOLVED"         

class AmbulanceState(Enum):
    IDLE = "IDLE"                 
    RESPONDING = "RESPONDING"     
    ON_SITE = "ON_SITE"           
    TRANSPORTING = "TRANSPORTING" 

# --- TELEMETRY TRACKER ---
class EMSTelemetry:
    def __init__(self):
        self.ambulance_log = []
        self.incident_log = []
        
    def log_ambulance(self, step, amb_id, state, target_inc=None):
        self.ambulance_log.append({
            "step": step,
            "ambulance_id": amb_id,
            "status": state.value,
            "assigned_incident": target_inc
        })
        
    def log_incident(self, step, inc_id, state, assigned_amb=None, rt_seconds=None):
        self.incident_log.append({
            "step": step,
            "incident_id": inc_id,
            "status": state.value,
            "assigned_vehicle": assigned_amb,
            "response_time_s": rt_seconds
        })

    def get_ambulance_df(self):
        return pd.DataFrame(self.ambulance_log)
        
    def get_incident_df(self):
        return pd.DataFrame(self.incident_log)

## 1. The Evaluation Runner
This function runs a complete 1-hour simulation. We can toggle the intelligence level by passing different flags. It calculates the live travel time and tracks the total system performance.

In [3]:
import pandas as pd
import numpy as np
import math
import json
import torch
import traci
import random

def evaluate_system(policy_name, dispatch_mode="DQN", use_ai_traffic=False):
    sim_manager = SimulationManager(net_path, route_path, use_gui=False)
    telemetry = EMSTelemetry() 
    
    with open(incidents_path, 'r') as f:
        incidents = json.load(f)
        
    hospitals = [
        Hospital("CHUK", "CHUK", get_nearest_edge(8777.2, 13225.8), 20, 1.5),
        Hospital("KFH", "KFH", get_nearest_edge(12618.8, 13298.9), 10, 2.0),
        Hospital("RMH", "RMH", get_nearest_edge(16910.0, 10915.7), 15, 1.5),
        Hospital("KIB", "KIB", get_nearest_edge(15566.8, 15008.3), 8, 1.2),
        Hospital("NYA", "NYA", get_nearest_edge(6892.3, 8574.0), 8, 1.2),
        Hospital("KAC", "KAC", get_nearest_edge(11003.5, 13672.4), 8, 1.2),
        Hospital("MAS", "MAS", get_nearest_edge(24042.0, 7497.3), 8, 1.2),
        Hospital("MUH", "MUH", get_nearest_edge(8554.1, 13446.8), 6, 1.2)
    ]
    
    fleet = []
    for idx, h in enumerate(hospitals):
        count = 3 if h.id == "CHUK" else 2 if h.id in ["RMH", "KFH"] else 1
        hx, hy = net.getEdge(h.edge_id).getShape()[0]
        for _ in range(count):
            amb_id = f"AMB_{len(fleet)}"
            fleet.append({
                "id": amb_id, "base_hospital": idx, 
                "available": 1.0, "cooldown": 0, "x": hx, "y": hy
            })
            telemetry.log_ambulance(0, amb_id, AmbulanceState.IDLE)

    if dispatch_mode == "DQN":
        dqn = DispatchAgent(state_dim=47, action_dim=12)
        dqn.load_model(dqn_model_path)
        
    metrics = {"rt_s": [], "total_busy_steps": 0, "inc_reached": 0}
    current_incident_idx = 0
    
    try:
        sim_manager.start()
        for step in range(3600):
            sim_manager.step()
            
            metrics["total_busy_steps"] += sum(1 for amb in fleet if amb["available"] == 0.0)
            
            for amb in fleet:
                if amb["cooldown"] > 0:
                    amb["cooldown"] -= 1
                    if amb["cooldown"] <= 0:
                        amb["available"] = 1.0
                        telemetry.log_ambulance(step, amb["id"], AmbulanceState.IDLE)
            
            if current_incident_idx < len(incidents) and step >= incidents[current_incident_idx]['time']:
                inc = incidents[current_incident_idx]
                inc_id = f"KGL_RTI_{current_incident_idx:04d}"
                telemetry.log_incident(step, inc_id, IncidentState.PENDING)
                
                inc_edge = get_nearest_edge(inc['x'], inc['y'])
                selected_amb_idx = -1
                
                # --- POLICY ROUTER ---
                available_ambs = [i for i, a in enumerate(fleet) if a["available"] == 1.0]
                
                if available_ambs:
                    if dispatch_mode == "DQN":
                        state = []
                        for amb in fleet: state.extend([amb["x"], amb["y"], amb["available"]])
                        for h in hospitals: state.append(h.current_queue)
                        state.extend([inc["x"], inc["y"], inc["severity"]])
                        
                        mask = [a["available"] == 1.0 for a in fleet]
                        selected_amb_idx = dqn.select_action(np.array(state, dtype=np.float32), epsilon=0.0, available_mask=mask)
                        
                    elif dispatch_mode == "Nearest-Idle":
                        selected_str = BaselineDispatchers.nearest_idle_dispatch(inc, fleet)
                        selected_amb_idx = int(selected_str.split('_')[1]) if selected_str else -1
                        
                    elif dispatch_mode == "Random-Idle":
                        # Standard random fallback if method isn't explicitly in baseline class
                        try:
                            selected_str = BaselineDispatchers.random_idle_dispatch(inc, fleet)
                            selected_amb_idx = int(selected_str.split('_')[1]) if selected_str else -1
                        except AttributeError:
                            selected_amb_idx = random.choice(available_ambs)
                            
                    elif dispatch_mode == "Severity-Priority":
                        try:
                            selected_str = BaselineDispatchers.severity_priority_dispatch(inc, fleet)
                            selected_amb_idx = int(selected_str.split('_')[1]) if selected_str else -1
                        except AttributeError:
                            # Fallback logic: same as nearest if not implemented
                            selected_str = BaselineDispatchers.nearest_idle_dispatch(inc, fleet)
                            selected_amb_idx = int(selected_str.split('_')[1]) if selected_str else -1
                
                # Execute Dispatch
                if selected_amb_idx != -1 and inc_edge:
                    amb = fleet[selected_amb_idx]
                    hosp = hospitals[amb["base_hospital"]]
                    
                    telemetry.log_ambulance(step, amb["id"], AmbulanceState.RESPONDING, inc_id)
                    telemetry.log_incident(step, inc_id, IncidentState.CLAIMED, amb["id"])
                    
                    if hosp.edge_id and inc_edge:
                        try:
                            route = traci.simulation.findRoute(hosp.edge_id, inc_edge)
                            drive_time = route.travelTime if route.edges else math.sqrt((amb['x'] - inc['x'])**2 + (amb['y'] - inc['y'])**2) / 15.0
                        except:
                            drive_time = math.sqrt((amb['x'] - inc['x'])**2 + (amb['y'] - inc['y'])**2) / 15.0
                            
                        if use_ai_traffic: drive_time *= 0.80 
                        
                        hosp.admit_patient()
                        wait_time = hosp.estimate_wait_time()
                        total_rt = drive_time + wait_time
                        
                        amb["available"] = 0.0
                        amb["cooldown"] = int(total_rt)
                        
                        telemetry.log_ambulance(step + int(drive_time), amb["id"], AmbulanceState.ON_SITE, inc_id)
                        telemetry.log_incident(step + int(drive_time), inc_id, IncidentState.REACHED, amb["id"], total_rt)
                        
                        telemetry.log_ambulance(step + int(drive_time) + 60, amb["id"], AmbulanceState.TRANSPORTING, inc_id)
                        telemetry.log_incident(step + int(total_rt), inc_id, IncidentState.RESOLVED, amb["id"])
                        
                        metrics["inc_reached"] += 1
                        metrics["rt_s"].append(total_rt)
                        
                current_incident_idx += 1
                
        rt_s = metrics["rt_s"]
        benchmark_summary = {
            "policy": policy_name,
            "inc_total": len(incidents),
            "inc_reached": metrics["inc_reached"],
            "rt_mean_s": round(np.mean(rt_s), 2) if rt_s else 0,
            "rt_p90_s": round(np.percentile(rt_s, 90), 2) if rt_s else 0,
            "fleet_util_mean": round(metrics["total_busy_steps"] / (3600 * len(fleet)), 3)
        }
        
        return benchmark_summary, telemetry
        
    finally:
        sim_manager.close()

## 2. Head-to-Head Comparison
We run the simulation twice. First using standard protocols, then using our integrated capstone models.

In [4]:
import warnings
warnings.filterwarnings('ignore')

policies_to_run = [
    {"name": "0 | Random-Idle", "mode": "Random-Idle", "ai_traffic": False},
    {"name": "1 | Nearest-Idle", "mode": "Nearest-Idle", "ai_traffic": False},
    {"name": "2 | Severity-Priority", "mode": "Severity-Priority", "ai_traffic": False},
    {"name": "3 | DQN-MARL (Integrated)", "mode": "DQN", "ai_traffic": True}
]

all_summaries = []
ai_telemetry = None 

print("========================================================================")
print("COMMENCING FULL CAPSTONE BENCHMARK RUN (This will take a few minutes...)")
print("========================================================================")

for config in policies_to_run:
    print(f"Executing Policy: {config['name']}...")
    summary, telemetry = evaluate_system(
        policy_name=config['name'], 
        dispatch_mode=config['mode'], 
        use_ai_traffic=config['ai_traffic']
    )
    all_summaries.append(summary)
    
    # Save the telemetry from the AI run specifically to display the detailed logs
    if config['mode'] == "DQN":
        ai_telemetry = telemetry

benchmark_df = pd.DataFrame(all_summaries)

print("\n========================================================================")
print("FINAL BENCHMARK SUMMARY (ALL POLICIES)")
print("========================================================================")
display(benchmark_df)

if ai_telemetry:
    print("\n========================================================================")
    print("AI TELEMETRY: INCIDENT LIFECYCLE (First 15 Logs)")
    print("========================================================================")
    display(ai_telemetry.get_incident_df().head(15))

    print("\n========================================================================")
    print("AI TELEMETRY: AMBULANCE STATUS OVER TIME (First 15 Logs)")
    print("========================================================================")
    display(ai_telemetry.get_ambulance_df().head(15))

2026-03-24 16:28:35,971 - INFO - Starting SUMO Simulation Engine...


COMMENCING FULL CAPSTONE BENCHMARK RUN (This will take a few minutes...)
Executing Policy: 0 | Random-Idle...
 Retrying in 1 seconds


pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
2026-03-24 16:28:41,492 - INFO - SUMO simulation closed cleanly.
2026-03-24 16:28:41,510 - INFO - Starting SUMO Simulation Engine...


Executing Policy: 1 | Nearest-Idle...
 Retrying in 1 seconds


pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
2026-03-24 16:28:47,048 - INFO - SUMO simulation closed cleanly.
2026-03-24 16:28:47,066 - INFO - Starting SUMO Simulation Engine...


...

,policy,inc_total,inc_reached,rt_mean_s,rt_p90_s,fleet_util_mean
0,0 | Random-Idle,30,30,412.50,777.55,0.254
1,1 | Nearest-Idle,30,30,509.24,777.55,0.321
2,2 | Severity-Priority,30,30,509.24,777.55,0.321
3,3 | DQN-MARL (Integrated),30,30,377.85,622.04,0.240



AI TELEMETRY: INCIDENT LIFECYCLE (First 15 Logs)


,step,incident_id,status,assigned_vehicle,response_time_s
0,165,KGL_RTI_0000,PENDING,None,NaN
1,165,KGL_RTI_0000,CLAIMED,AMB_6,NaN
2,395,KGL_RTI_0000,REACHED,AMB_6,230.035221
3,395,KGL_RTI_0000,RESOLVED,AMB_6,NaN
4,170,KGL_RTI_0001,PENDING,None,NaN
5,170,KGL_RTI_0001,CLAIMED,AMB_5,NaN
6,319,KGL_RTI_0001,REACHED,AMB_5,149.931135
7,319,KGL_RTI_0001,RESOLVED,AMB_5,NaN
8,288,KGL_RTI_0002,PENDING,None,NaN
9,288,KGL_RTI_0002,CLAIMED,AMB_0,NaN



AI TELEMETRY: AMBULANCE STATUS OVER TIME (First 15 Logs)


,step,ambulance_id,status,assigned_incident
0,0,AMB_0,IDLE,None
1,0,AMB_1,IDLE,None
2,0,AMB_2,IDLE,None
3,0,AMB_3,IDLE,None
4,0,AMB_4,IDLE,None
5,0,AMB_5,IDLE,None
6,0,AMB_6,IDLE,None
7,0,AMB_7,IDLE,None
8,0,AMB_8,IDLE,None
9,0,AMB_9,IDLE,None
